In [3]:
# ============================================================
# IF 공공데이터 전체 API 확인용
# 목적: API 호출 → 응답 확인 → DataFrame → 컬럼/NULL/샘플 확인
# 아직 전처리 / JOIN / DB 적재 안 함
# ============================================================

import os
import requests
import pandas as pd
import xml.etree.ElementTree as ET

from pathlib import Path
from dotenv import load_dotenv
from IPython.display import display


# ============================================================
# 1. .env 찾기
# ============================================================

current = Path.cwd().resolve()

env_candidates = [
    current / ".env",
    current / "data" / ".env",
    current.parent / ".env",
    current.parent / "data" / ".env",
]

env_path = None

for path in env_candidates:
    if path.exists():
        env_path = path
        break

if env_path is None:
    raise FileNotFoundError(".env 파일을 찾을 수 없습니다.")

load_dotenv(env_path, override=True)

print("ENV 위치:", env_path)


# ============================================================
# 2. .env 값 가져오기
# ============================================================

API_KEY = (
    os.getenv("DATA_GO_KR_API_KEY")
    or os.getenv("PUBLIC_DATA_API_KEY")
)

JOB_BSN = (
    os.getenv("JOB_BSN_API_URL")
    or os.getenv("JOB_BSN_API_BASE_URL")
)

SENURI = (
    os.getenv("SENURI_API_URL")
    or os.getenv("SENURI_API_BASE_URL")
)

LTC_INSTITUTION = (
    os.getenv("LTC_INSTITUTION_API_URL")
    or os.getenv("LTC_INST_API_BASE_URL")
)

ODSN = (
    os.getenv("ODSN_CODE_API_URL")
    or os.getenv("ODSN_CODE_API_BASE_URL")
)

LTC_STATUS = (
    os.getenv("LTC_STATUS_API_URL")
    or os.getenv("ODCLOUD_3051420_URL")
)

LTC_REGION = (
    os.getenv("LTC_REGION_API_URL")
    or os.getenv("ODCLOUD_15072696_URL")
)

ACCIDENT_AGE = (
    os.getenv("ACCIDENT_AGE_API_URL")
    or os.getenv("ODCLOUD_15064490_URL")
)

ACCIDENT_INDUSTRY = (
    os.getenv("ACCIDENT_INDUSTRY_API_URL")
    or os.getenv("ODCLOUD_15064475_URL")
)

ACCIDENT_REGION = (
    os.getenv("ACCIDENT_REGION_API_URL")
    or os.getenv("ODCLOUD_15084656_URL")
)

ACCIDENT_SIZE = (
    os.getenv("ACCIDENT_SIZE_API_URL")
    or os.getenv("ODCLOUD_15064486_URL")
)


print("API KEY 등록:", bool(API_KEY))

if not API_KEY:
    raise RuntimeError("API KEY를 .env에서 찾지 못했습니다.")


# ============================================================
# 3. .env에 등록된 데이터 확인
# ============================================================

datasets = {
    "01 노인 구인정보": SENURI,
    "02 장기요양기관": LTC_INSTITUTION,
    "03 자립형 노인일자리": JOB_BSN,
    "04 노인사회활동 코드": ODSN,
    "05 장기요양 인정현황": LTC_STATUS,
    "06 장기요양 지역현황": LTC_REGION,
    "07 연령별 재해": ACCIDENT_AGE,
    "08 산업별 재해": ACCIDENT_INDUSTRY,
    "09 지역별 재해": ACCIDENT_REGION,
    "10 규모별 재해": ACCIDENT_SIZE,
}

print("\n================ 등록 데이터 ================\n")

for name, url in datasets.items():
    print(name)
    print("URL:", url)
    print()


# ============================================================
# 4. DataFrame 확인 함수
# ============================================================

def show_dataframe(name, df):

    print("\n")
    print("=" * 70)
    print(name)
    print("=" * 70)

    print("행/열:", df.shape)

    print("\n[전체 컬럼]")
    print(df.columns.tolist())

    print("\n[샘플 데이터]")
    display(df.head(5))

    print("\n[컬럼 분석]")

    profile = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "NULL": df.isna().sum(),
        "NULL_%": (df.isna().mean() * 100).round(2),
        "unique": df.nunique(dropna=True),
    })

    display(profile)

    print("\n[전체 중복 행]")
    print(df.duplicated().sum())

    print("\n[컬럼별 샘플값]")

    for col in df.columns:

        values = (
            df[col]
            .dropna()
            .astype(str)
            .unique()[:5]
            .tolist()
        )

        print(col, "→", values)


# ============================================================
# 5. 결과 저장용
# ============================================================

results = {}

status_results = []


# ============================================================
# 6. XML API
# ============================================================

xml_apis = {

    "01 노인 구인정보": {
        "url": f"{SENURI}/getJobList" if SENURI else None,
        "params": {
            "serviceKey": API_KEY,
            "pageNo": 1,
            "numOfRows": 100,
        }
    },

    "02 장기요양기관": {
        "url": (
            f"{LTC_INSTITUTION}/getLtcInsttSeachList02"
            if LTC_INSTITUTION
            else None
        ),
        "params": {
            "serviceKey": API_KEY,
            "pageNo": 1,
            "numOfRows": 100,
        }
    },

    "03 자립형 노인일자리": {
        "url": (
            f"{JOB_BSN}/getJobBsnRecruitList"
            if JOB_BSN
            else None
        ),
        "params": {
            "serviceKey": API_KEY,
            "pageNo": 1,
            "numOfRows": 100,
        }
    },
}


# ============================================================
# 7. XML API 실행
# ============================================================

for name, config in xml_apis.items():

    print("\n\n")
    print("#" * 70)
    print(name)
    print("#" * 70)

    url = config["url"]

    if not url:

        print("❌ .env에 URL 없음")

        status_results.append({
            "데이터셋": name,
            "HTTP": None,
            "상태": "URL 없음",
            "행": None,
            "열": None,
        })

        continue

    try:

        response = requests.get(
            url,
            params=config["params"],
            timeout=30
        )

        print("HTTP:", response.status_code)
        print(
            "Content-Type:",
            response.headers.get("Content-Type")
        )

        print("\n[응답 앞부분]")
        print(response.text[:700])

        if response.status_code != 200:

            print("\n❌ HTTP 오류")

            status_results.append({
                "데이터셋": name,
                "HTTP": response.status_code,
                "상태": "HTTP 오류",
                "행": None,
                "열": None,
            })

            continue


        # XML 파싱
        root = ET.fromstring(response.content)


        # API 응답 코드 확인
        result_code = root.findtext(".//resultCode")
        result_msg = root.findtext(".//resultMsg")
        total_count = root.findtext(".//totalCount")

        print("\nresultCode:", result_code)
        print("resultMsg:", result_msg)
        print("totalCount:", total_count)


        # item 찾기
        items = root.findall(".//item")

        print("조회 item:", len(items))


        if len(items) == 0:

            print("\n⚠️ API 호출은 됐지만 데이터가 0건입니다.")
            print("검색조건/지역코드/필수 파라미터 확인 필요")

            status_results.append({
                "데이터셋": name,
                "HTTP": response.status_code,
                "상태": "0건 / 조건 확인",
                "행": 0,
                "열": 0,
            })

            continue


        # XML → DataFrame
        rows = []

        for item in items:

            row = {}

            for child in item:
                row[child.tag] = child.text

            rows.append(row)


        df = pd.DataFrame(rows)

        results[name] = df


        status_results.append({
            "데이터셋": name,
            "HTTP": response.status_code,
            "상태": "성공",
            "행": len(df),
            "열": len(df.columns),
        })


        show_dataframe(name, df)


    except Exception as e:

        print("\n❌ 실행 오류")
        print(type(e).__name__)
        print(e)

        status_results.append({
            "데이터셋": name,
            "HTTP": None,
            "상태": f"실행 오류: {type(e).__name__}",
            "행": None,
            "열": None,
        })


# ============================================================
# 8. 노인사회활동 코드 API
# ============================================================

print("\n\n")
print("#" * 70)
print("04 노인사회활동 코드")
print("#" * 70)

if ODSN:

    try:

        response = requests.get(
            ODSN,
            params={
                "serviceKey": API_KEY,
                "pageNo": 1,
                "numOfRows": 10,
            },
            timeout=30
        )

        print("HTTP:", response.status_code)

        print("\n[응답]")
        print(response.text[:1500])


        if response.status_code == 200:

            try:

                root = ET.fromstring(response.content)

                items = root.findall(".//item")

                if items:

                    rows = []

                    for item in items:

                        row = {}

                        for child in item:
                            row[child.tag] = child.text

                        rows.append(row)

                    df = pd.DataFrame(rows)

                    results["04 노인사회활동 코드"] = df

                    show_dataframe(
                        "04 노인사회활동 코드",
                        df
                    )

                    status_results.append({
                        "데이터셋": "04 노인사회활동 코드",
                        "HTTP": 200,
                        "상태": "성공",
                        "행": len(df),
                        "열": len(df.columns),
                    })

                else:

                    print("⚠️ item 없음")

                    status_results.append({
                        "데이터셋": "04 노인사회활동 코드",
                        "HTTP": 200,
                        "상태": "데이터 없음/operation 확인",
                        "행": 0,
                        "열": 0,
                    })

            except ET.ParseError:

                print("⚠️ XML 파싱 실패")

        else:

            print("\n⚠️ 현재 URL/operation 확인 필요")

            status_results.append({
                "데이터셋": "04 노인사회활동 코드",
                "HTTP": response.status_code,
                "상태": "URL/operation 확인",
                "행": None,
                "열": None,
            })


    except Exception as e:

        print(type(e).__name__, e)

else:

    print("❌ .env URL 없음")


# ============================================================
# 9. odcloud API
# ============================================================

odcloud_apis = {

    "05 장기요양 인정현황": LTC_STATUS,

    "06 장기요양 지역현황": LTC_REGION,

    "07 연령별 재해": ACCIDENT_AGE,

    "08 산업별 재해": ACCIDENT_INDUSTRY,

    "09 지역별 재해": ACCIDENT_REGION,

    "10 규모별 재해": ACCIDENT_SIZE,
}


# ============================================================
# 10. odcloud 실행
# ============================================================

for name, url in odcloud_apis.items():

    print("\n\n")
    print("#" * 70)
    print(name)
    print("#" * 70)

    if not url:

        print("❌ .env URL 없음")

        status_results.append({
            "데이터셋": name,
            "HTTP": None,
            "상태": "URL 없음",
            "행": None,
            "열": None,
        })

        continue


    try:

        response = requests.get(
            url,
            params={
                "serviceKey": API_KEY,
                "page": 1,
                "perPage": 100,
            },
            timeout=30
        )


        print("HTTP:", response.status_code)

        print(
            "Content-Type:",
            response.headers.get("Content-Type")
        )

        print("\n[응답 앞부분]")
        print(response.text[:700])


        # HTTP 실패
        if response.status_code != 200:

            print("\n⚠️ 현재 URL만으로 데이터 조회 실패")
            print("Swagger의 실제 resource path 확인 필요")

            status_results.append({
                "데이터셋": name,
                "HTTP": response.status_code,
                "상태": "resource path 확인",
                "행": None,
                "열": None,
            })

            continue


        # JSON
        result = response.json()

        print("\n[JSON KEY]")
        print(result.keys())


        data = result.get("data", [])


        if not data:

            print("\n⚠️ data 없음")

            status_results.append({
                "데이터셋": name,
                "HTTP": 200,
                "상태": "data 없음",
                "행": 0,
                "열": 0,
            })

            continue


        df = pd.DataFrame(data)

        results[name] = df


        status_results.append({
            "데이터셋": name,
            "HTTP": 200,
            "상태": "성공",
            "행": len(df),
            "열": len(df.columns),
        })


        show_dataframe(name, df)


    except Exception as e:

        print("\n❌ 실행 오류")
        print(type(e).__name__)
        print(e)

        status_results.append({
            "데이터셋": name,
            "HTTP": None,
            "상태": f"실행 오류: {type(e).__name__}",
            "행": None,
            "열": None,
        })


# ============================================================
# 11. 전체 성공/실패 결과
# ============================================================

print("\n\n")
print("=" * 80)
print("전체 API 테스트 결과")
print("=" * 80)

status_df = pd.DataFrame(status_results)

display(status_df)


# ============================================================
# 12. 실제 DataFrame 생성 성공 데이터
# ============================================================

print("\n")
print("=" * 80)
print("DataFrame 생성 성공 데이터")
print("=" * 80)


if len(results) == 0:

    print("성공 데이터 없음")

else:

    success_summary = []

    for name, df in results.items():

        success_summary.append({
            "데이터셋": name,
            "행": len(df),
            "열": len(df.columns),
            "컬럼": ", ".join(df.columns.tolist()),
        })

    success_df = pd.DataFrame(success_summary)

    display(success_df)


# ============================================================
# 13. 성공한 데이터셋별 전체 컬럼 프로파일
# ============================================================

for name, df in results.items():

    print("\n\n")
    print("=" * 80)
    print(name)
    print("=" * 80)

    rows = []

    for col in df.columns:

        sample_values = (
            df[col]
            .dropna()
            .astype(str)
            .unique()[:5]
            .tolist()
        )

        rows.append({
            "컬럼": col,
            "dtype": str(df[col].dtype),
            "NULL": df[col].isna().sum(),
            "NULL_%": round(
                df[col].isna().mean() * 100,
                2
            ),
            "unique": df[col].nunique(
                dropna=True
            ),
            "샘플값": sample_values,
        })


    column_profile = pd.DataFrame(rows)

    display(column_profile)


# ============================================================
# 끝
# ============================================================

print("\n")
print("=" * 80)
print("API 원본 데이터 확인 완료")
print("=" * 80)

print("""
지금 확인할 것

1. 어떤 API가 성공했는지
2. 각각 몇 행 × 몇 열인지
3. 전체 컬럼이 무엇인지
4. NULL이 많은 컬럼이 무엇인지
5. 실제 샘플값이 무엇인지
6. 중복이 있는지

아직 하지 않을 것

- NULL 삭제
- 이상값 삭제
- 컬럼 삭제
- 데이터 JOIN
- DB 테이블 생성
- PostgreSQL 적재

→ 전체 데이터 구조를 확인한 다음 전처리 규칙을 정한다.
""")

ENV 위치: /Users/jeong-yelim/IF_portfolio/data/.env
API KEY 등록: True

================ 등록 데이터 ================

01 노인 구인정보
URL: https://apis.data.go.kr/B552474/SenuriService

02 장기요양기관
URL: https://apis.data.go.kr/B550928/searchLtcInsttService02

03 자립형 노인일자리
URL: https://apis.data.go.kr/B552474/JobBsnInfoService

04 노인사회활동 코드
URL: https://apis.data.go.kr/B552474/OdsnCodeInquiryService2

05 장기요양 인정현황
URL: https://api.odcloud.kr/api/3051420/v1

06 장기요양 지역현황
URL: https://api.odcloud.kr/api/15072696/v1

07 연령별 재해
URL: https://api.odcloud.kr/api/15064486/v1

08 산업별 재해
URL: https://api.odcloud.kr/api/15084656/v1

09 지역별 재해
URL: https://api.odcloud.kr/api/15064475/v1

10 규모별 재해
URL: https://api.odcloud.kr/api/15064490/v1




######################################################################
01 노인 구인정보
######################################################################
HTTP: 403
Content-Type: application/xml

[응답 앞부분]
<?xml version="1.0" encoding="UTF-8"?>
<OpenAPI_ServiceResponse>
<cmmM

,데이터셋,HTTP,상태,행,열
0,01 노인 구인정보,403,HTTP 오류,None,None
1,02 장기요양기관,403,HTTP 오류,None,None
2,03 자립형 노인일자리,403,HTTP 오류,None,None
3,04 노인사회활동 코드,400,URL/operation 확인,None,None
4,05 장기요양 인정현황,404,resource path 확인,None,None
5,06 장기요양 지역현황,404,resource path 확인,None,None
6,07 연령별 재해,404,resource path 확인,None,None
7,08 산업별 재해,404,resource path 확인,None,None
8,09 지역별 재해,404,resource path 확인,None,None
9,10 규모별 재해,404,resource path 확인,None,None




DataFrame 생성 성공 데이터
성공 데이터 없음


API 원본 데이터 확인 완료

지금 확인할 것

1. 어떤 API가 성공했는지
2. 각각 몇 행 × 몇 열인지
3. 전체 컬럼이 무엇인지
4. NULL이 많은 컬럼이 무엇인지
5. 실제 샘플값이 무엇인지
6. 중복이 있는지

아직 하지 않을 것

- NULL 삭제
- 이상값 삭제
- 컬럼 삭제
- 데이터 JOIN
- DB 테이블 생성
- PostgreSQL 적재

→ 전체 데이터 구조를 확인한 다음 전처리 규칙을 정한다.

